# Regresión — experimentar con otros modelos

En el cuaderno anterior usamos un modelo de regresión lineal simple. Aquí probamos modelos más complejos para intentar superar ese baseline.

La celda siguiente repite la carga y la división de datos, con la **misma semilla** (`random_state=0`) para que la comparación sea justa: mismos datos de entrenamiento y de prueba que en el baseline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Cargar el dataset
bike_data = pd.read_csv('../datasets/daily-bike-share.csv')
bike_data['day'] = pd.DatetimeIndex(bike_data['dteday']).day

# Separar features y labels
X, y = bike_data[['season', 'mnth', 'holiday', 'weekday', 'workingday', 'weathersit',
                  'temp', 'atemp', 'hum', 'windspeed']].values, bike_data['rentals'].values

# Misma división que en el baseline
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

print(f'Entrenamiento: {X_train.shape[0]} filas · Prueba: {X_test.shape[0]} filas')

## Una función para no repetir código

Los cuatro modelos que vamos a probar se evalúan exactamente igual. En vez de copiar y pegar el bloque de métricas cuatro veces, lo encapsulamos una vez.

Este es el patrón importante: cuando pruebas varios modelos, lo único que cambia es el **estimador**; todo lo demás (entrenar, predecir, medir, graficar) es idéntico.

In [ ]:
resultados = []

def evaluar(model, nombre, mostrar_grafico=True):
    """Entrena, evalúa y grafica un modelo. Devuelve sus métricas."""
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)

    print(f'--- {nombre} ---')
    print(f'MAE : {mae:.4f}')
    print(f'MSE : {mse:.4f}')
    print(f'RMSE: {rmse:.4f}')
    print(f'R2  : {r2:.4f}\n')

    if mostrar_grafico:
        plt.scatter(y_test, predictions)
        plt.xlabel('Labels reales')
        plt.ylabel('Labels predichos')
        plt.title(nombre)
        z = np.polyfit(y_test, predictions, 1)
        p = np.poly1d(z)
        plt.plot(y_test, p(y_test), color='magenta')
        plt.show()

    resultados.append({'modelo': nombre, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
    return model

## Baseline: regresión lineal

Primero reproducimos el baseline del cuaderno anterior, para tenerlo en la misma tabla comparativa.

In [ ]:
from sklearn.linear_model import LinearRegression

evaluar(LinearRegression(), 'LinearRegression (baseline)')

## Qué otros algoritmos existen

El algoritmo de regresión lineal tiene cierta capacidad predictiva, pero hay muchas familias de algoritmos de regresión:

- **Lineales**: no solo `LinearRegression` (que técnicamente es *mínimos cuadrados ordinarios*), sino variantes como **Lasso** y **Ridge**.
- **Basados en árboles**: construyen un árbol de decisión para llegar a una predicción.
- **De conjunto (ensemble)**: combinan las salidas de varios modelos base para mejorar la generalización.

### Otro algoritmo lineal: Lasso

Cambiar de algoritmo es literalmente cambiar el estimador. Todo lo demás queda igual.

In [ ]:
from sklearn.linear_model import Lasso

evaluar(Lasso(), 'Lasso')

### Árbol de decisión

Alternativa a los modelos lineales: examinar las features en una serie de evaluaciones, cada una de las cuales produce una **rama** del árbol según el valor de la feature. Al final de cada serie de ramas hay nodos hoja con el valor predicho.

Entrenemos uno y veamos su estructura.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, export_text

modelo_arbol = DecisionTreeRegressor(random_state=0)
modelo_arbol.fit(X_train, y_train)

print(f'Profundidad del árbol completo: {modelo_arbol.get_depth()} niveles')
print(f'Nodos hoja: {modelo_arbol.get_n_leaves()}\n')

# El árbol completo tiene cientos de nodos: imprimirlo entero no se puede leer.
# Mostramos solo los 3 primeros niveles para ver la lógica de las ramas.
print(export_text(modelo_arbol, max_depth=3))

> **Nota**: el ejercicio original imprime el árbol completo. Como este árbol crece sin poda hasta cientos de niveles, aquí lo limitamos a 3 niveles — lo interesante es ver **cómo ramifica**, no leer miles de líneas.
>
> Fíjate en la lógica: cada línea es una pregunta del tipo "¿la feature X está por debajo de tal valor?", y bajando por las ramas se llega a un valor predicho.

Ahora evaluémoslo.

In [ ]:
evaluar(DecisionTreeRegressor(random_state=0), 'DecisionTreeRegressor')

El modelo basado en árbol no mejora significativamente al lineal. ¿Qué más se puede probar?

### Algoritmos de conjunto (ensemble)

Combinan varios estimadores base para producir un modelo mejor. Hay dos estrategias:

- **Bagging**: construir muchos modelos de forma **independiente** y promediar sus resultados.
- **Boosting**: construir modelos en **secuencia**, donde cada uno intenta corregir el error del anterior.

Empecemos con un **Random Forest**, que aplica una función de promedio sobre múltiples árboles de decisión (bagging).

In [ ]:
from sklearn.ensemble import RandomForestRegressor

evaluar(RandomForestRegressor(random_state=0), 'RandomForestRegressor')

Ahora un algoritmo de **boosting**: `GradientBoostingRegressor`. Igual que Random Forest construye muchos árboles, pero en vez de construirlos todos de forma independiente y promediar, **cada árbol se construye sobre las salidas del anterior**, intentando reducir la pérdida de forma incremental.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

evaluar(GradientBoostingRegressor(random_state=0), 'GradientBoostingRegressor')

## Comparativa

Ahora que están todos en la misma tabla, se ve cuál gana:

In [ ]:
df_resultados = pd.DataFrame(resultados).sort_values('R2', ascending=False).reset_index(drop=True)
df_resultados.round(4)

In [ ]:
# Comparación visual del R2
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(df_resultados['modelo'], df_resultados['R2'], color='steelblue')
ax.set_xlabel('R²')
ax.set_title('Comparación de modelos (mayor es mejor)')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
for i, v in enumerate(df_resultados['R2']):
    ax.text(v + 0.005, i, f'{v:.4f}', va='center')
plt.tight_layout()
plt.show()

## Resumen

Probamos varios algoritmos de regresión sobre los mismos datos. Los de conjunto suelen ganar a los lineales y al árbol simple, porque combinan muchos modelos en vez de confiar en uno solo.

En el cuaderno `03-optimizacion-modelos.ipynb` vamos a **afinar** el mejor de estos, ajustando sus hiperparámetros y preprocesando los datos.